# DESARROLLO DE MODELOS DE MACHINE LEARNING

Desarrollado Por: 
* Daniel Arturo Martinez Morales 
* Luis Miguel Ortega Cañate

A continuacion se presenta el desarrollo de la fase de entrenamiento  y test de los modelos  de machine learning  para el desarrollo de un Sistema Clasificador de Resultados de Análisis de Aceite Usado en Motores de Maquinaria Pesada.

**Nota:** Para el desarrollo de esta fase se tiene en cuenta ls fase de exploración de los datos desarrolado previamente en el Noteboock llamado "Exploracion de datos".

## Balanceo de Muestras 

Durante la fase de exploración de datos se identificó un desbalance significativo
en la variable objetivo `Assigned Condition Rating`, donde la clase Normal concentra
la gran mayoría de las observaciones, mientras que las clases Advertencia y Crítico
representan una proporción considerablemente menor del total de muestras.

Este desbalance es inherente al dominio del problema: en operación normal, la mayor
parte de los componentes de una flota minera se encuentran en condición saludable,
y los eventos de advertencia o falla crítica son relativamente poco frecuentes.

Sin embargo, este desbalance introduce
los siguientes riesgos si no es tratado:

- El modelo tiende a clasificar la mayoría de muestras como Normal para maximizar
  el accuracy global, ignorando las clases minoritarias.
- Se genera un sesgo hacia la clase mayoritaria, reduciendo drásticamente el
  desempeño  para las clases Advertencia y Crítico, que son precisamente las
  de mayor valor operativo para el negocio.
- Las métricas globales como el Accuracy resultan engañosas, ocultando el
  bajo desempeño real del modelo sobre las condiciones anómalas.

Detectar una condición Crítica que el modelo clasifica como Normal representa un Falso Negativo de alto costo operativo, asociado directamente a las pérdidas de ~500 kUSD/año documentadas en el planteamiento del problema.


In [5]:
import pandas as pd
data_cleaned = pd.read_excel("../Exploración/Analisis De Aceite Flotas Mineras.xlsx")

In [6]:
print("=== Distribución original de clases ===")
print(data_cleaned['Assigned Condition Rating'].value_counts())
print(data_cleaned['Assigned Condition Rating'].value_counts(normalize=True)
      .mul(100).round(2).astype(str) + ' %')

=== Distribución original de clases ===
Assigned Condition Rating
Normal      3607
Warning      522
Critical     154
Name: count, dtype: int64
Assigned Condition Rating
Normal      84.22 %
Warning     12.19 %
Critical      3.6 %
Name: proportion, dtype: object


In [12]:
import pandas as pd
import numpy as np


# CALIFICACIÓN SACODE - Generación de Límites Condenatorios (Metodología Noria)


## Definición de Predictores según cada Categoría
Salud = data_cleaned[['Boro (ppm)', 'Calcio (ppm)', 'Cinc (ppm)', 'Fosforado (ppm)',
                       'Magnesio (ppm)', 'Molibdeno (ppm)', 'Nitración JOAP (Abs/cm)',
                       'Oxidación JOAP (Abs/cm)', 'Sulfatación JOAP (Abs/cm)', 'Sodio (ppm)']]

Desgaste = data_cleaned[['Aluminio (ppm)', 'Cobre (ppm)', 'Cromo (ppm)', 'Estaño (ppm)',
                          'Hierro (ppm)', 'Plomo (ppm)',
                          'Oxidación JOAP (Abs/cm)']]

Contaminacion = data_cleaned[['Agua (%)', 'Dilución por combustible (%)',
                               'Hollín JOAP (Abs/cm)', 'Silicio (ppm)',
                               'Sodio (ppm)', 'Aluminio (ppm)']]



def calcular_limites(df: pd.DataFrame, 
                     factor_warning: float = 2.0, 
                     factor_critico: float = 3.0) -> pd.DataFrame:

    stats = pd.DataFrame({
        'media'           : df.mean(),
        'sigma'           : df.std(),
        'lim_warn_sup'    : df.mean() + factor_warning * df.std(),
        'lim_warn_inf'    : df.mean() - factor_warning * df.std(),
        'lim_critico_sup' : df.mean() + factor_critico * df.std(),
        'lim_critico_inf' : df.mean() - factor_critico * df.std(),
    })

    stats['lim_warn_inf']    = stats['lim_warn_inf'].clip(lower=0)
    stats['lim_critico_inf'] = stats['lim_critico_inf'].clip(lower=0)

    return stats



def clasificar_predictor(valor: float, 
                         lim_warn_sup: float, 
                         lim_critico_sup: float) -> str:

    if valor <= lim_warn_sup:
        return 'Normal'
    elif valor <= lim_critico_sup:
        return 'Advertencia'
    else:
        return 'Crítico'



# Función: Calificación SACODE por categoría (criterio de peor caso)


def calificar_categoria(df: pd.DataFrame, limites: pd.DataFrame) -> pd.Series:

    orden_severidad = {'Normal': 0, 'Advertencia': 1, 'Crítico': 2}
    inv_orden       = {v: k for k, v in orden_severidad.items()}

    calificaciones = pd.DataFrame(index=df.index)

    for col in df.columns:
        lim_w = limites.loc[col, 'lim_warn_sup']
        lim_c = limites.loc[col, 'lim_critico_sup']
        calificaciones[col] = df[col].apply(
            lambda x: clasificar_predictor(x, lim_w, lim_c)
        )

    # Peor caso: máximo nivel de severidad entre todos los predictores por fila
    peor_caso = calificaciones.map(lambda x: orden_severidad[x]).max(axis=1)

    return peor_caso.map(inv_orden)



# Cálculo de Límites por Categoría


limites_salud        = calcular_limites(Salud)
limites_desgaste     = calcular_limites(Desgaste)
limites_contaminacion = calcular_limites(Contaminacion)

print("=== Límites Salud ===")
print(limites_salud.round(3), "\n")

print("=== Límites Desgaste ===")
print(limites_desgaste.round(3), "\n")

print("=== Límites Contaminación ===")
print(limites_contaminacion.round(3), "\n")

# Asignación de Calificación SACODE al dataset


data_cleaned['SACODE_Salud']         = calificar_categoria(Salud,         limites_salud)
data_cleaned['SACODE_Desgaste']      = calificar_categoria(Desgaste,      limites_desgaste)
data_cleaned['SACODE_Contaminacion'] = calificar_categoria(Contaminacion, limites_contaminacion)

=== Límites Salud ===
                              media    sigma  lim_warn_sup  lim_warn_inf  \
Boro (ppm)                   57.419   39.573       136.565         0.000   
Calcio (ppm)               2045.069  690.602      3426.273       663.865   
Cinc (ppm)                 1351.095  149.697      1650.489      1051.702   
Fosforado (ppm)            1179.651   78.502      1336.655      1022.647   
Magnesio (ppm)              241.588  142.551       526.691         0.000   
Molibdeno (ppm)              53.587   14.047        81.680        25.494   
Nitración JOAP (Abs/cm)       5.791    1.364         8.518         3.063   
Oxidación JOAP (Abs/cm)      13.620    2.617        18.855         8.385   
Sulfatación JOAP (Abs/cm)    18.457    2.613        23.683        13.231   
Sodio (ppm)                   5.874   47.363       100.600         0.000   

                           lim_critico_sup  lim_critico_inf  
Boro (ppm)                         176.139            0.000  
Calcio (ppm)     

In [13]:

# Limite Warning sodio
limites_salud.loc['Sodio (ppm)', 'lim_warn_sup'] = 40
# Limite Critico sodio
limites_salud.loc['Sodio (ppm)', 'lim_critico_sup'] = 100
# Limite Warning  Oxidacion JOAP
limites_salud.loc['Oxidación JOAP (Abs/cm)', 'lim_warn_sup'] = 19
# Limite Critico  Oxidacion JOAP
limites_salud.loc['Oxidación JOAP (Abs/cm)', 'lim_critico_sup'] = 21
# Limite Warning Cobre
limites_desgaste.loc['Cobre (ppm)', 'lim_warn_sup'] = 6
# Limite Critico Cobre
limites_desgaste.loc['Cobre (ppm)', 'lim_critico_sup'] = 8
#Limite Warning Plomo
limites_desgaste.loc['Plomo (ppm)', 'lim_warn_sup'] = 4
#Limite Critico Plomo
limites_desgaste.loc['Plomo (ppm)', 'lim_critico_sup'] = 6

In [15]:
data_cleaned['SACODE_Salud']         = calificar_categoria(Salud,         limites_salud)
data_cleaned['SACODE_Desgaste']      = calificar_categoria(Desgaste,      limites_desgaste)
data_cleaned['SACODE_Contaminacion'] = calificar_categoria(Contaminacion, limites_contaminacion)

In [16]:
data_cleaned['SACODE_General'] = data_cleaned[['SACODE_Salud', 'SACODE_Desgaste', 'SACODE_Contaminacion']].apply(
    lambda x: 'Crítico' if 'Crítico' in x.values else ('Advertencia' if 'Advertencia' in x.values else 'Normal'),
    axis=1
)

### Estrategia de Balanceo

Para mitigar el desbalance se aplica la técnica SMOTE, que genera muestras sintéticas
para las clases minoritarias interpolando entre observaciones reales del espacio
de características, en lugar de simplemente duplicar registros existentes.

In [10]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split


# Variables predictoras y variable objetivo
X = data_cleaned.drop(columns=['Assigned Condition Rating',
                                'ACR_Homologado',
                                'Fault Effect',
                                'SACODE_Salud',
                                'SACODE_Desgaste',
                                'SACODE_Contaminacion',
                                'SACODE_General','Flota','Rule Based Rating'])

y = data_cleaned['Assigned Condition Rating']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y       
)

print("=== Distribución antes de SMOTE (Train) ===")
print(y_train.value_counts())

# Aplicación de SMOTE exclusivamente sobre el conjunto de entrenamiento
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("\n=== Distribución después de SMOTE (Train) ===")
print(pd.Series(y_train_bal).value_counts())
print("\n=== Conjunto de prueba (sin modificar) ===")
print(y_test.value_counts())

=== Distribución antes de SMOTE (Train) ===
Assigned Condition Rating
Normal      2885
Warning      418
Critical     123
Name: count, dtype: int64

=== Distribución después de SMOTE (Train) ===
Assigned Condition Rating
Normal      2885
Warning     2885
Critical    2885
Name: count, dtype: int64

=== Conjunto de prueba (sin modificar) ===
Assigned Condition Rating
Normal      722
Warning     104
Critical     31
Name: count, dtype: int64


c:\Users\Daniel Martinez\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Daniel Martinez\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Daniel Martinez\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Daniel Martinez\AppData\Local\Pro

## Parte V: Calificación SACODE (Salud - Contaminación - Desgaste)

* En esta etapa se incluyen nuevas features al dataset previamente depurado.
* Se agregan 3 nuevas variables al dataframe: **Salud**, **Contaminación** y **Desgaste**.
* Cada una de estas variables puede tomar 3 estados posibles: **Normal**, **Advertencia** y **Crítico**.
* La calificación se construye mediante la generación de **límites condenatorios** para los predictores
  asociados a cada categoría SACODE, siguiendo la metodología de Noria.
* Los límites se calculan a partir de la media y la desviación estándar de cada predictor,
  definidos de la siguiente manera:

  * **Límite Warning:**
    $$L_{warning} = \bar{x} \pm 2\sigma$$

  * **Límite Crítico:**
    $$L_{critico} = \bar{x} \pm 3\sigma$$

* La lógica de clasificación por predictor opera de la siguiente forma:
  * Si el valor observado se encuentra dentro del **Límite Warning** → estado **Normal**
  * Si el valor observado supera el **Límite Warning** pero no el **Límite Crítico** → estado **Advertencia**
  * Si el valor observado supera el **Límite Crítico** → estado **Crítico**

* La calificación final de cada categoría SACODE (Salud, Contaminación, Desgaste)
  corresponde al estado más severo registrado entre todos sus predictores asociados
  (criterio de peor caso).

In [20]:
Salud_train = X_train_bal[['Boro (ppm)', 'Calcio (ppm)', 'Cinc (ppm)', 'Fosforado (ppm)',
                       'Magnesio (ppm)', 'Molibdeno (ppm)', 'Nitración JOAP (Abs/cm)',
                       'Oxidación JOAP (Abs/cm)', 'Sulfatación JOAP (Abs/cm)', 'Sodio (ppm)']]

Desgaste_train = X_train_bal[['Aluminio (ppm)', 'Cobre (ppm)', 'Cromo (ppm)', 'Estaño (ppm)',
                          'Hierro (ppm)', 'Plomo (ppm)',
                          'Oxidación JOAP (Abs/cm)']]

Contaminacion_train = X_train_bal[['Agua (%)', 'Dilución por combustible (%)',
                               'Hollín JOAP (Abs/cm)', 'Silicio (ppm)',
                               'Sodio (ppm)', 'Aluminio (ppm)']]


In [25]:
X_train_bal['SACODE_Salud']         = calificar_categoria(Salud_train,         limites_salud)
X_train_bal['SACODE_Desgaste']      = calificar_categoria(Desgaste_train,      limites_desgaste)
X_train_bal['SACODE_Contaminacion'] = calificar_categoria(Contaminacion_train, limites_contaminacion)
X_train_bal['SACODE_General'] = X_train_bal[['SACODE_Salud', 'SACODE_Desgaste', 'SACODE_Contaminacion']].apply(
    lambda x: 'Crítico' if 'Crítico' in x.values else ('Advertencia' if 'Advertencia' in x.values else 'Normal'),
    axis=1
)

In [23]:
Salud_test = X_test[['Boro (ppm)', 'Calcio (ppm)', 'Cinc (ppm)', 'Fosforado (ppm)',
                       'Magnesio (ppm)', 'Molibdeno (ppm)', 'Nitración JOAP (Abs/cm)',
                       'Oxidación JOAP (Abs/cm)', 'Sulfatación JOAP (Abs/cm)', 'Sodio (ppm)']]

Desgaste_test = X_test[['Aluminio (ppm)', 'Cobre (ppm)', 'Cromo (ppm)', 'Estaño (ppm)',
                          'Hierro (ppm)', 'Plomo (ppm)',
                          'Oxidación JOAP (Abs/cm)']]

Contaminacion_test = X_test[['Agua (%)', 'Dilución por combustible (%)',
                               'Hollín JOAP (Abs/cm)', 'Silicio (ppm)',
                               'Sodio (ppm)', 'Aluminio (ppm)']]

In [27]:
X_test['SACODE_Salud']         = calificar_categoria(Salud_test,         limites_salud)
X_test['SACODE_Desgaste']      = calificar_categoria(Desgaste_test,      limites_desgaste)
X_test['SACODE_Contaminacion'] = calificar_categoria(Contaminacion_test, limites_contaminacion)
X_test['SACODE_General'] = X_test[['SACODE_Salud', 'SACODE_Desgaste', 'SACODE_Contaminacion']].apply(
    lambda x: 'Crítico' if 'Crítico' in x.values else ('Advertencia' if 'Advertencia' in x.values else 'Normal'),
    axis=1
)

In [26]:
X_train_bal

,Unnamed: 0,Boro (ppm),Calcio (ppm),Cinc (ppm),Fosforado (ppm),Magnesio (ppm),Molibdeno (ppm),Agua (%),Dilución por combustible (%),Hollín JOAP (Abs/cm),...,Aluminio (ppm),Cobre (ppm),Cromo (ppm),Estaño (ppm),Hierro (ppm),Plomo (ppm),SACODE_Salud,SACODE_Desgaste,SACODE_Contaminacion,SACODE_General
0,3667,0.200000,1341.000000,1239.302808,1162.037844,125.019635,60.100000,0.0,0.000000,0.000000,...,1.000000,1.000000,0.300000,0.000000,3.000000,2.200000,Normal,Normal,Normal,Normal
1,2691,47.520000,3530.000000,1419.662325,1152.073179,21.250000,34.230000,0.0,0.000000,2.000000,...,2.370000,1.000000,0.510000,0.000000,6.000000,1.000000,Advertencia,Advertencia,Advertencia,Advertencia
2,2472,48.050000,3443.000000,1328.234553,1135.981777,20.980000,34.360000,0.0,0.000000,0.000000,...,1.260000,0.190000,0.660000,0.044622,3.040000,0.000000,Advertencia,Normal,Normal,Advertencia
3,2838,45.980000,3444.966354,1246.839800,1038.889783,21.030000,35.020000,0.0,0.000000,0.000000,...,1.000000,1.000000,0.090000,0.240000,3.000000,1.000000,Advertencia,Normal,Normal,Advertencia
4,3978,0.100000,1179.000000,1232.105806,1162.340280,123.585327,49.100000,0.0,0.000000,1.000000,...,1.000000,1.000000,0.300000,0.000000,3.000000,1.100000,Normal,Normal,Normal,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8650,1876,38.214101,1828.747241,1428.106841,1225.723046,286.485725,63.933353,0.0,0.076639,22.968316,...,0.972631,3.296721,0.452734,0.000000,11.660776,7.966251,Normal,Crítico,Advertencia,Crítico
8651,2777,46.063606,3466.994788,1298.710451,1059.843691,25.396557,35.769331,0.0,0.000000,3.000000,...,1.825259,1.000000,1.354073,0.000000,10.808320,1.000000,Crítico,Advertencia,Normal,Crítico
8652,2270,41.546815,3383.957678,1129.628176,1045.484504,64.727263,33.270121,0.0,0.000000,8.248069,...,1.583504,1.710203,0.648760,0.019323,5.152558,9.996660,Crítico,Crítico,Normal,Crítico
8653,227,64.999477,1808.709779,1400.045463,1383.639820,375.952647,61.816959,0.0,0.068462,22.311189,...,0.995595,0.448671,0.284895,0.000000,9.805072,0.413567,Advertencia,Normal,Advertencia,Advertencia


### Desarrollo de Modelos de Aprendizaje 

* Una vez balencados la muestras de la base de datos suministrados por el cliente se proccede a generar los Pipeline de entrenamiento y de test de los algoritmos candidatos a ser utilizados en este proyecto los cuales son: 

1. Modelo de Ensaamble de Random Forest
2. Modelo de Ensamble de Gradient Boosting
3. Modelo de  Regresion Logistica 
4. Modelo de Support Vector Machine (SVM)
5. Modelos de Redes Neuronales  Multi-Layer Perceptron


A continuación, se muestra el pipeline desarrollado para  los modelos escogidos.

In [ ]:


import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.neural_network  import MLPClassifier
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics         import (classification_report, 
                                     confusion_matrix,
                                     ConfusionMatrixDisplay,
                                     f1_score, recall_score,
                                     roc_auc_score)
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier

os.environ['LOKY_MAX_CPU_COUNT'] = '4'
warnings.filterwarnings('ignore')


# Columnas a excluir del entrenamiento
COLS_EXCLUIR = [
    'Assigned Condition Rating',
    'ACR_Homologado',
    'Fault Effect',
    'SACODE_Salud',
    'SACODE_Desgaste',
    'SACODE_Contaminacion',
    'SACODE_General',
    'Flota'                      # Categórica identificadora — no predictora
]

# Features y target
X = data_cleaned.drop(columns=[c for c in COLS_EXCLUIR 
                                if c in data_cleaned.columns])
y = data_cleaned['Assigned Condition Rating']

# Codificación del target
label_encoder  = LabelEncoder()
label_encoder.fit(['Normal', 'Warning', 'Critical'])
y_encoded      = label_encoder.transform(y)
CLASES         = label_encoder.classes_          # ['Critical', 'Normal', 'Warning']
CLASES_ES      = ['Crítico', 'Normal', 'Advertencia']

print(f"Shape features    : {X.shape}")
print(f"Clases target     : {label_encoder.classes_}")
print(f"Distribución target:\n{y.value_counts()}\n")

# ==============================================================================
# 2. DIVISIÓN TRAIN / TEST (post-SMOTE ya aplicado en celda anterior)
#    X_train_bal, y_train_bal  → conjunto balanceado para entrenamiento
#    X_test,      y_test       → conjunto original para evaluación
# ==============================================================================

# Re-codificación del target balanceado y de prueba
y_train_enc = label_encoder.transform(y_train_bal)
y_test_enc  = label_encoder.transform(y_test)

print(f"Train balanceado  : {X_train_bal.shape} | Clases: {np.unique(y_train_enc, return_counts=True)}")
print(f"Test original     : {X_test.shape}      | Clases: {np.unique(y_test_enc,  return_counts=True)}\n")

# ==============================================================================
# 3. DEFINICIÓN DE MODELOS — PIPELINES CON PREPROCESAMIENTO
# ==============================================================================
# Nota: LR, SVM y MLP requieren escalamiento.
#       RF, XGBoost y LightGBM son invariantes al escalamiento.

modelos = {

    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            multi_class  = 'multinomial',
            solver       = 'lbfgs',
            C            = 1.0,
            max_iter     = 1000,
            class_weight = 'balanced',
            random_state = 42
        ))
    ]),

    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(
            n_estimators     = 300,
            max_depth        = None,
            min_samples_leaf = 5,
            class_weight     = 'balanced',
            n_jobs           = -1,
            random_state     = 42
        ))
    ]),

    'XGBoost': Pipeline([
        ('clf', XGBClassifier(
            n_estimators     = 300,
            max_depth        = 5,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            use_label_encoder= False,
            eval_metric      = 'mlogloss',
            n_jobs           = -1,
            random_state     = 42
        ))
    ]),

    'LightGBM': Pipeline([
        ('clf', LGBMClassifier(
            n_estimators     = 300,
            max_depth        = 5,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            class_weight     = 'balanced',
            n_jobs           = -1,
            random_state     = 42,
            verbose          = -1
        ))
    ]),

    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(
            kernel       = 'rbf',
            C            = 10,
            gamma        = 'scale',
            class_weight = 'balanced',
            probability  = True,
            random_state = 42
        ))
    ]),

    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(
            hidden_layer_sizes = (128, 64, 32),
            activation         = 'relu',
            solver             = 'adam',
            learning_rate_init = 0.001,
            max_iter           = 500,
            early_stopping     = True,
            validation_fraction= 0.1,
            random_state       = 42
        ))
    ])
}

# ==============================================================================
# 4. VALIDACIÓN CRUZADA ESTRATIFICADA (5-Fold sobre Train balanceado)
# ==============================================================================

print("=" * 65)
print("VALIDACIÓN CRUZADA — 5-Fold Estratificado (Train Balanceado)")
print("=" * 65)

CV            = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
METRICAS_CV   = ['f1_weighted', 'recall_weighted', 'accuracy']
resultados_cv = {}

for nombre, pipeline in modelos.items():
    print(f"\n  Entrenando: {nombre}...")
    cv_result = cross_validate(
        estimator  = pipeline,
        X          = X_train_bal,
        y          = y_train_enc,
        cv         = CV,
        scoring    = METRICAS_CV,
        n_jobs     = -1,
        return_train_score = False
    )
    resultados_cv[nombre] = {
        'F1 Weighted (CV)'      : cv_result['test_f1_weighted'].mean(),
        'F1 Weighted Std'       : cv_result['test_f1_weighted'].std(),
        'Recall Weighted (CV)'  : cv_result['test_recall_weighted'].mean(),
        'Accuracy (CV)'         : cv_result['test_accuracy'].mean(),
    }
    print(f"    F1 Weighted : {resultados_cv[nombre]['F1 Weighted (CV)']:.4f} "
          f"(±{resultados_cv[nombre]['F1 Weighted Std']:.4f})")
    print(f"    Recall      : {resultados_cv[nombre]['Recall Weighted (CV)']:.4f}")
    print(f"    Accuracy    : {resultados_cv[nombre]['Accuracy (CV)']:.4f}")

df_cv = pd.DataFrame(resultados_cv).T.sort_values('F1 Weighted (CV)', ascending=False)
print("\n=== Resumen Validación Cruzada ===")
print(df_cv.round(4))

# ==============================================================================
# 5. ENTRENAMIENTO FINAL Y EVALUACIÓN SOBRE TEST
# ==============================================================================

print("\n" + "=" * 65)
print("EVALUACIÓN SOBRE CONJUNTO DE PRUEBA (Distribución Original)")
print("=" * 65)

resultados_test  = {}
modelos_fit      = {}

for nombre, pipeline in modelos.items():
    print(f"\n{'─'*50}")
    print(f"  Modelo: {nombre}")
    print(f"{'─'*50}")

    # Entrenamiento final sobre el conjunto balanceado completo
    pipeline.fit(X_train_bal, y_train_enc)
    modelos_fit[nombre] = pipeline

    # Predicciones sobre test (distribución original)
    y_pred     = pipeline.predict(X_test)
    y_pred_proba = (pipeline.predict_proba(X_test) 
                    if hasattr(pipeline.named_steps.get('clf', pipeline), 'predict_proba') 
                    else None)

    # Métricas
    f1_critico = f1_score(y_test_enc, y_pred, 
                          labels=[label_encoder.transform(['Critical'])[0]], 
                          average=None)[0]
    recall_critico = recall_score(y_test_enc, y_pred,
                                  labels=[label_encoder.transform(['Critical'])[0]],
                                  average=None)[0]
    auc = (roc_auc_score(y_test_enc, y_pred_proba, 
                         multi_class='ovr', average='weighted')
           if y_pred_proba is not None else np.nan)

    resultados_test[nombre] = {
        'F1 Weighted (Test)'   : f1_score(y_test_enc, y_pred, average='weighted'),
        'F1 Crítico (Test)'    : f1_critico,
        'Recall Crítico (Test)': recall_critico,
        'AUC-ROC (Test)'       : auc,
        'Accuracy (Test)'      : (y_pred == y_test_enc).mean()
    }

    print(classification_report(
        y_test_enc, y_pred,
        target_names = label_encoder.classes_
    ))

df_test = pd.DataFrame(resultados_test).T.sort_values('Recall Crítico (Test)', ascending=False)
print("\n=== Resumen Evaluación Test ===")
print(df_test.round(4))

# ==============================================================================
# 6. MATRICES DE CONFUSIÓN — TODOS LOS MODELOS
# ==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes      = axes.flatten()

for i, (nombre, pipeline) in enumerate(modelos_fit.items()):
    y_pred = pipeline.predict(X_test)
    cm     = confusion_matrix(y_test_enc, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix = cm,
        display_labels   = label_encoder.classes_
    )
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f'{nombre}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Predicción')
    axes[i].set_ylabel('Real')

plt.suptitle('Matrices de Confusión — Conjunto de Prueba\n(Distribución Original)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ==============================================================================
# 7. COMPARATIVA VISUAL DE MÉTRICAS
# ==============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metricas_plot = {
    'F1 Weighted (Test)'   : '#3498db',
    'Recall Crítico (Test)': '#e74c3c',
    'AUC-ROC (Test)'       : '#2ecc71'
}

for ax, (metrica, color) in zip(axes, metricas_plot.items()):
    df_plot = df_test[metrica].sort_values(ascending=True)
    bars    = ax.barh(df_plot.index, df_plot.values, color=color, edgecolor='white')
    ax.set_xlim(0, 1.05)
    ax.set_title(metrica, fontsize=11, fontweight='bold')
    ax.set_xlabel('Score')
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
    ax.axvline(x=0.7, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)

plt.suptitle('Comparativa de Modelos — Conjunto de Prueba',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ==============================================================================
# 8. SELECCIÓN DEL MEJOR MODELO
# ==============================================================================

mejor_modelo_nombre = df_test['Recall Crítico (Test)'].idxmax()
mejor_modelo        = modelos_fit[mejor_modelo_nombre]

print("\n" + "=" * 65)
print(f"  MEJOR MODELO: {mejor_modelo_nombre}")
print(f"  Recall Crítico : {df_test.loc[mejor_modelo_nombre, 'Recall Crítico (Test)']:.4f}")
print(f"  F1 Weighted    : {df_test.loc[mejor_modelo_nombre, 'F1 Weighted (Test)']:.4f}")
print(f"  AUC-ROC        : {df_test.loc[mejor_modelo_nombre, 'AUC-ROC (Test)']:.4f}")
print("=" * 65)